# cGAN — Conditional Generative Adversarial Nets (2014)

_Papers · Generative Models_

**Feed the class label into both the generator and the discriminator, and you can ask a GAN to generate a chosen digit on demand.**

---

This notebook is a practice scaffold for the **cGAN — Conditional Generative Adversarial Nets (2014)** lesson. We build the conditional GAN one step at a time — the maths sanity-check, the label-aware nets, the data, the sampler, and the adversarial loop — then watch it generate a digit we choose. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Sanity-check the conditional optimum on paper

Before writing any network, we pin down what "trained" should look like. For a fixed class `y`, the optimal discriminator is `D*(x|y) = p_data(x|y) / (p_data(x|y) + p_g(x|y))`, exactly the plain-GAN formula but now *per class*. At the global optimum the generator matches the data for every class, the value function bottoms out at `-log4`, and the discriminator's BCE loss settles near `2·log2`. These constants are our targets for the run below.

In [ ]:
import math
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
K = 10                                          # MNIST has 10 classes
Z = 64                                          # noise vector length

# Sanity-check the worked example: conditional optimal D and the global optimum.
pdy = 0.6                                        # p_data(x|y) at this point
pgy = 0.2                                        # p_g(x|y) at this point
Dstar = pdy / (pdy + pgy)                        # D*(x|y) = p_data / (p_data + p_g)

print("worked example (per condition y):")
print("  D*(x|y) = %.2f/%.2f = %.4f" % (pdy, pdy + pgy, Dstar))       # 0.75
print("  log D*(x|y)          = %.4f" % math.log(Dstar))            # -0.2877
print("  fake term log(1-0.3) = %.4f" % math.log(1 - 0.3))          # -0.3567
print("  global optimum -log4 = %.4f" % (-math.log(4)))            # -1.3863
print("  D-loss at equilibrium 2*log2 = %.4f" % (2 * math.log(2)))   # 1.3863

### Step 2 — Build the label-aware generator and discriminator

This is the whole idea of a cGAN: the class label is concatenated into the input of *both* networks. The generator takes `[noise z | one-hot y]` and emits a 784-pixel image; the discriminator takes `[image x | one-hot y]` and outputs one real/fake logit. The `+K` on each first layer is the room for that one-hot. We give each net an Adam optimiser and use `BCEWithLogitsLoss`, so the sigmoid lives inside the loss.

In [ ]:
def onehot(y):                                  # integer labels -> one-hot rows
    return torch.eye(K, device=y.device)[y]


class G(nn.Module):                             # [noise z | label y] -> 784 pixels in [-1,1]
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(Z + K, 256),              # +K: room for the one-hot label
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Tanh())

    def forward(self, z, y):
        inputs = torch.cat([z, onehot(y)], 1)   # concatenate noise with the label
        return self.net(inputs)


class D(nn.Module):                             # [image x | label y] -> 1 real/fake logit
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784 + K, 512),            # +K: the same one-hot label
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1))                  # sigmoid lives inside the BCE loss

    def forward(self, x, y):
        inputs = torch.cat([x, onehot(y)], 1)   # concatenate image with the label
        return self.net(inputs)


gen = G().to(device)
dis = D().to(device)
bce = nn.BCEWithLogitsLoss()
optG = torch.optim.Adam(gen.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(dis.parameters(), lr=2e-4, betas=(0.5, 0.999))

### Step 3 — Load MNIST and a preview helper

We load MNIST and normalise pixels to `[-1, 1]` so they match the generator's `tanh` output. The `preview` helper takes a *requested* digit class, runs the generator on a fixed noise vector with that label, and renders the 28×28 result as ASCII art — that fixed noise lets us watch the same seed turn into different digits purely because we change the label.

In [ ]:
# MNIST (torchvision is preinstalled in Colab). Scale to [-1,1] to match tanh.
tfm = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(data, batch_size=128, shuffle=True, drop_last=True)

# Fixed noise so the ONLY thing changing across previews is the requested label.
fixed_z = torch.randn(1, Z, device=device)


def preview(want):                              # want = the digit class we REQUEST
    gen.eval()
    with torch.no_grad():
        y = torch.tensor([want], device=device)
        img = gen(fixed_z, y).view(28, 28).cpu()
    gen.train()
    img = (img + 1) / 2                         # back from [-1,1] to [0,1] for display
    print("--- requested digit:", want, "---")
    for r in range(0, 28, 2):
        row = "".join(" .:-=+*#%@"[min(9, int(img[r, c].clamp(0, 1) * 9))] for c in range(28))
        print(row)

### Step 4 — Run the conditional adversarial loop

Each step runs two updates. The **discriminator** judges real images under their *true* labels and fake images under the *same* label they were generated for — matched `(x, y)` pairs are what teach it the per-class real/fake boundary. The **generator** then uses the non-saturating loss, trying to make `D` call its class-`yf` fakes "real for `yf`". We `detach()` the fakes in the D-step so that update never moves G. After each epoch we request a different digit so we can see the conditioning take hold.

In [ ]:
def train(epochs=3):
    step = 0
    for ep in range(epochs):
        for x, y in loader:
            x = x.view(x.size(0), -1).to(device)
            y = y.to(device)
            m = x.size(0)
            real_t = torch.ones(m, 1, device=device)
            fake_t = torch.zeros(m, 1, device=device)

            # (a) DISCRIMINATOR step: reals under TRUE labels, fakes under the SAME label.
            z = torch.randn(m, Z, device=device)
            yf = torch.randint(0, K, (m,), device=device)   # labels we generate fakes for
            fake = gen(z, yf).detach()                       # detach: this step must NOT move G
            lossD = bce(dis(x, y), real_t) + bce(dis(fake, yf), fake_t)
            optD.zero_grad()
            lossD.backward()
            optD.step()

            # (b) GENERATOR step: non-saturating -- make D call class-yf fakes "real for yf".
            z = torch.randn(m, Z, device=device)
            yf = torch.randint(0, K, (m,), device=device)
            lossG = bce(dis(gen(z, yf), yf), real_t)
            optG.zero_grad()
            lossG.backward()
            optG.step()

            if step % 600 == 0:
                print("step %5d   D loss %.3f   G loss %.3f" % (step, lossD.item(), lossG.item()))
            step += 1
        preview(want=ep % K)                                 # ask for a different digit each epoch


print("\nbefore training (requesting a 3):")
preview(3)
train(epochs=3)
print("\nNow generate any digit on demand:")
for d in [0, 5, 9]:
    preview(d)
print("\nD loss settles near 2*log2 = 1.386 PER CLASS; the requested digit should emerge.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

### Step 5 — Define the unconditional baseline for the ablation

To prove the label is what buys controllable generation, we define an *unconditional* generator whose only input is noise — no label slot anywhere. Trained with the plain-GAN loop, it can still make digits, but you cannot ask it for a specific class: which digit appears is whatever `z` happens to map to. That contrast is the point of the ablation, explored in the practice problems.

In [ ]:
class Gu(nn.Module):                            # unconditional: noise only, no label
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(Z, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Tanh())

    def forward(self, z):
        return self.net(z)


# Train Gu with the plain-GAN loop (no y anywhere); requesting a class is impossible --
# its only input is noise, so the digit is whatever z maps to. That is the point of the ablation.

## Visualize the data & results

_Does conditioning on the label let us generate a CHOSEN class on demand — and what happens when we ablate the label?_

### Step 1 — A tiny 3-class toy dataset and the net builder

MNIST is slow, so we reproduce the same conditional-vs-unconditional contrast on a 2-D toy: three Gaussian blobs, one per class, standing in for digit classes. `make_nets(conditional)` returns a generator/discriminator pair whose input width grows by `K` only when we want conditioning — the single switch that turns the cGAN into a plain GAN.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# 3-class toy data: each class is a Gaussian blob at a distinct center (a stand-in for
# MNIST's digit classes). Same loop as the MNIST notebook, small enough to run in seconds.
K = 3
CENTERS = torch.tensor([[2.0, 0.0], [-1.0, 1.7], [-1.0, -1.7]])
Z = 8


def real_batch(m):
    y = torch.randint(0, K, (m,))
    x = torch.randn(m, 2) * 0.30 + CENTERS[y]   # blob around the class center
    return x, y


def onehot(y):
    return torch.eye(K)[y]


def make_nets(conditional):
    c = K if conditional else 0                 # add K input dims only when conditioning
    G = nn.Sequential(nn.Linear(Z + c, 64), nn.ReLU(), nn.Linear(64, 2))
    D = nn.Sequential(nn.Linear(2 + c, 64), nn.ReLU(), nn.Linear(64, 1))
    return G, D

### Step 2 — Train either variant with one shared loop

`train(conditional)` runs the identical adversarial loop for both variants; the only difference is whether the one-hot label is concatenated onto the discriminator inputs and the generator input. `gen_class` then samples from a trained generator: for the conditional one we set the requested class, for the unconditional one we just feed noise and take whatever comes out.

In [ ]:
def train(conditional, steps=1500):
    torch.manual_seed(1)
    G, D = make_nets(conditional)
    bce = nn.BCEWithLogitsLoss()
    optD = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.5, 0.999))
    optG = torch.optim.Adam(G.parameters(), lr=2e-3, betas=(0.5, 0.999))
    m = 128
    hist = []
    for s in range(steps):
        x, y = real_batch(m)
        oh = onehot(y)
        z = torch.randn(m, Z)

        # Discriminator step: concatenate the label only in the conditional variant.
        gin = torch.cat([z, oh], 1) if conditional else z
        dr = torch.cat([x, oh], 1) if conditional else x
        fake = G(gin).detach()
        df = torch.cat([fake, oh], 1) if conditional else fake
        lossD = bce(D(dr), torch.ones(m, 1)) + bce(D(df), torch.zeros(m, 1))
        optD.zero_grad()
        lossD.backward()
        optD.step()

        # Generator step.
        z = torch.randn(m, Z)
        gin = torch.cat([z, oh], 1) if conditional else z
        g = G(gin)
        din = torch.cat([g, oh], 1) if conditional else g
        lossG = bce(D(din), torch.ones(m, 1))
        optG.zero_grad()
        lossG.backward()
        optG.step()

        if s % 50 == 0:
            hist.append((s, round(lossD.item(), 4)))
    return G, hist


def gen_class(G, k, n, conditional):
    z = torch.randn(n, Z)
    if conditional:
        labels = onehot(torch.full((n,), k))    # request class k
        return G(torch.cat([z, labels], 1)).detach()
    return G(z).detach()                        # unconditional: noise only

### Step 3 — Compare conditional vs ablated generation

We train both variants and probe them. For the conditional generator we request each class and measure how close its samples land to that class center (`err`) and what fraction fall in the right blob (`purity`). For the ablated one we pool many samples: it produces a single average blob that sits far from any one class, so it cannot honour a request. Purity near 1.0 vs a large error is the whole story.

In [ ]:
Gc, dlc = train(True)
Gu, _ = train(False)

print("CONDITIONAL -- request each class, mean & purity:")
for k in range(K):
    s = gen_class(Gc, k, 2000, True)
    err = (s.mean(0) - CENTERS[k]).norm().item()
    nearest = torch.cdist(s, CENTERS).argmin(1)         # which center each sample is closest to
    purity = (nearest == k).float().mean().item()
    print(" class %d: err %.3f  purity %.3f" % (k, err, purity))

pool = gen_class(Gu, 0, 6000, False)
mu = pool.mean(0)
print("ABLATED -- single mean %s; dist to each requested class:" % [round(v, 3) for v in mu.tolist()])
for k in range(K):
    print(" 'asking' class %d: err %.3f" % (k, (mu - CENTERS[k]).norm().item()))

print("Conditional D-loss history:", dlc[:10])
# Conditional steers to the chosen class (purity ~1.0); ablated cannot (one blob, err ~1.98).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The conditioning ablation. Take your working cGAN and remove the label from BOTH nets (drop the
            one-hot from the inputs and shrink the first layers back by $K$). Keep everything else identical and
            retrain. Then "request" class 2. What can the ablated generator do, and what does that prove the label
            was buying you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Strip $y$: make $G$'s input just $z$ and $D$'s input just $x$ (the plain paper-gan). — _An honest ablation changes exactly one thing &mdash; whether the nets see the label._
- Retrain, then try to generate class 2: there is no label slot to set, so $G$ just emits its usual mixture of all classes. — _With no condition in $G$, the only input is noise; which class appears is whatever $z$ maps to &mdash; uncontrollable._
- Measure: generate a pool and compare its single mean to each class center; it sits in the middle, far from any one class. — _An unconditional $G$ covers the whole data distribution at once; it cannot collapse onto a requested class._
- Conclude the label conditioning is exactly the steering wheel &mdash; remove it and you lose on-demand class generation. — _This is the paper's whole contribution, demonstrated by its absence._

**Answer:** The ablated generator has no way to receive the request: its only input is noise, so it
                 produces an uncontrolled mixture of all classes and ignores "class 2" entirely. In our toy run the
                 conditional generator hits purity 1.000 per requested class and a mean error of about
                 0.38 to the right class center, while the ablated generator's single blob sits ~1.98
                 away from any requested class (it can only ever produce one overall average). This proves that
                 concatenating the label into both nets is precisely what buys controllable, on-demand class
                 generation &mdash; the entire point of the paper.

</details>

**Problem 2.** Your worked example had $p_{\text{data}}(x|y)=0.6$ and $p_g(x|y)=0.2$, giving $D^*(x|y)=0.75$.
            Suppose for that class the generator has now matched the data at this point, so $p_g(x|y)=0.6$. What is
            $D^*(x|y)$, and what does it say about a converged cGAN per class?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into the conditional Prop. 1: $D^*(x|y) = \dfrac{0.6}{0.6+0.6} = \dfrac{0.6}{1.2} = 0.5$. — _When real and fake per-class densities are equal at $x$, the class-aware detector cannot tell them apart._
- Note that if $p_g(\cdot|y)=p_{\text{data}}(\cdot|y)$ for every class, then $D^*=0.5$ everywhere. — _The discriminator is reduced to a coin flip within each class &mdash; the equilibrium of the conditional game._
- Map to the loss: each BCE term becomes $-\log 0.5 = \log 2 \approx 0.693$, so $D$'s total settles near $2\log 2 \approx 1.386 = -\log 4$. — _Same plateau as a plain GAN, now reached class by class._

**Answer:** $D^*(x|y) = 0.6/(0.6+0.6) = 0.5$. Once the generator matches the data for that class, the
                 best class-aware discriminator outputs $0.5$ &mdash; it can only guess within the class. If
                 this holds for every class, $D^*=0.5$ everywhere and $D$'s loss settles near $2\log 2\approx 1.386$
                 ($-\log 4$), not $0$. A converged cGAN reaches the GAN equilibrium independently per class.

</details>

**Problem 3.** In the discriminator step you generate a fake for class 3 but, by a bug, hand $D$ the label "7" when
            judging it. Why does this break conditioning, and what is the correct pairing?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall $D(x|y)$ answers "is $x$ a real example of class $y$?" — _The label is part of what $D$ checks; it is not a passive tag._
- See the bug: $D$ is now asked whether a class-3 fake is a real "7" &mdash; it learns the wrong association between samples and labels. — _The training signal that ties "this image" to "this class" is corrupted._
- Fix it: judge each fake under the SAME label it was generated with (here, "3"). — _Only matched (sample, label) pairs teach $D$ the per-class real-vs-fake boundary, which is what forces $G$ to obey the label._

**Answer:** $D(x|y)$ scores realism for the given label. Generating a class-3 fake but judging it as
                 a "7" teaches $D$ a wrong sample-label association and removes the pressure on $G$ to match the
                 requested class, so control degrades. The correct pairing is to feed $D$ the same label the
                 sample was generated under &mdash; matched $(x, y)$ pairs on both the real and fake sides.

</details>